In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")
test = pd.read_csv("/kaggle/input/digit-recognizer/test.csv")

# Prepare Data

In [ ]:
Y_train = train['label']

X_train = train.drop('label', axis = 1)
X_train = X_train / 255.0
X_train = X_train.values.reshape(-1, 28, 28, 1)

X_test = test/ 255.0
X_test = X_test.values.reshape(-1, 28, 28, 1)

In [ ]:
import seaborn as sns
g = sns.countplot(x = Y_train) #Distribution of labels

# One-hot Encoding

In [ ]:
from keras.utils import to_categorical
Y_train = to_categorical(Y_train, num_classes = 10)


# Preview Image

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize = (15, 4.5)) 
for i in range(30):
    plt.subplot(3, 10, i + 1)
    plt.imshow(X_train[i], cmap = plt.cm.binary)
    plt.axis('off')
plt.subplots_adjust(wspace = -0.1, hspace = -0.1)
plt.show()

# Data Augmentation
Data augmentation is a technique in machine learning to artificially increase the size and diversity of a dataset by creating modified copies of existing data samples

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range = 10,
    zoom_range = 0.1,
    width_shift_range = 0.1,
    height_shift_range = 0.1
)

Preview augmented images

In [ ]:
X_train3 = X_train[9,].reshape((1,28,28,1))
Y_train3 = Y_train[9,].reshape((1,10))

plt.figure(figsize=(15,4.5))
for i in range(30):  
    plt.subplot(3, 10, i+1)
    X_train2, Y_train2 = datagen.flow(X_train3,Y_train3).__next__()
    plt.imshow(X_train2[0].reshape((28,28)),cmap=plt.cm.binary)
    plt.axis('off')
    if i==9: X_train3 = X_train[11,].reshape((1,28,28,1))
    if i==19: X_train3 = X_train[18,].reshape((1,28,28,1))
plt.subplots_adjust(wspace=-0.1, hspace=-0.1)
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, test_size = 0.2, random_state = 30)

# Build CNN

In [ ]:
from keras.models import Sequential
from keras.layers import Conv2D, BatchNormalization, MaxPool2D, Flatten, Dense, Dropout

model = Sequential()
model.add(Conv2D(32, kernel_size = 3, activation = 'relu', input_shape = (28, 28, 1)))
model.add(BatchNormalization())
model.add(Conv2D(32, kernel_size = 3, activation = 'relu'))
model.add(BatchNormalization())
model.add(Conv2D(32, kernel_size = 5, strides = 2, padding = 'same', activation = 'relu'))
model.add(BatchNormalization())
model.add(Dropout(0.4))

model.add(Conv2D(64, kernel_size = 3, activation = 'relu'))
model.add(BatchNormalization())
model.add(Conv2D(64, kernel_size = 3, activation = 'relu'))
model.add(BatchNormalization())
model.add(Conv2D(64, kernel_size = 5, strides = 2, padding = 'same', activation = 'relu'))
model.add(BatchNormalization())
model.add(Dropout(0.4))

model.add(Conv2D(128, kernel_size = 4, activation = 'relu', input_shape = (28, 28, 1)))
model.add(BatchNormalization())
model.add(Flatten())
model.add(Dropout(0.4))
model.add(Dense(10, activation = 'softmax'))



In [ ]:
from keras.optimizers import RMSprop
optimizer = RMSprop(learning_rate = 0.001, rho = 0.9, epsilon = 1e-08)
model.compile(optimizer = optimizer , loss = "categorical_crossentropy", metrics = ["accuracy"])

In [ ]:
from keras.callbacks import ReduceLROnPlateau

learning_rate_reduction = ReduceLROnPlateau(monitor = 'val_acc',
                            patience = 3,
                            verbose = 1,
                            factor = 0.5,
                            min_lr = 0.000005)

# Train CNNs¶

In [ ]:
history = model.fit(datagen.flow(X_train, Y_train, batch_size = 128),
                   epochs  = 15,
                   validation_data = (X_val, Y_val),
                   verbose = 2,
                   steps_per_epoch = X_train.shape[0]//128,
                   callbacks = [learning_rate_reduction])

# Plot the results


In [ ]:
# plots for accuracy and Loss with epochs

error = pd.DataFrame(history.history)

plt.figure(figsize=(18,5),dpi=200)
sns.set_style('darkgrid')

plt.subplot(121)
plt.title('Cross Entropy Loss',fontsize=15)
plt.xlabel('Epochs',fontsize=12)
plt.ylabel('Loss',fontsize=12)
plt.plot(error['loss'])
plt.plot(error['val_loss'])

plt.subplot(122)
plt.title('Classification Accuracy',fontsize=15)
plt.xlabel('Epochs',fontsize=12)
plt.ylabel('Accuracy',fontsize=12)
plt.plot(error['accuracy'])
plt.plot(error['val_accuracy'])

plt.show()

In [ ]:
results = model.predict(X_test)

# select the indix with the maximum probability
results = np.argmax(results,axis = 1)

results = pd.Series(results,name="Label")

# Submission

In [ ]:
# csv file output for submission

sub = pd.concat([pd.Series(range(1,28001),name = "ImageId"),results],axis = 1)

sub.to_csv("submission.csv",index=False)